In [ ]:
import pandas as pd
import json
import ast

NMRBank = pd.read_csv('../../raw_data/NMRBank/NMRBank.csv')
with open('../../raw_data/NMRBank/NMRBank_hnmr.json', 'r') as f:
    thread_results = json.load(f)

# Process HNMR data in NMRBank with LLM

In [ ]:
import multiprocessing as mp
import time
import random
from openai import OpenAI
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
import queue
import threading
from typing import List, Dict, Any
from tqdm import tqdm
from volcenginesdkarkruntime import Ark

# Alternative approach using threading for I/O bound operations
class DeepSeekThreadProcessor:
    def __init__(self, api_tokens: List[str], max_workers: int = None):
        """
        Initialize the thread processor with multiple DeepSeek API tokens
        """
        self.api_tokens = api_tokens
        self.max_workers = max_workers or len(api_tokens)
        self.available_tokens = queue.Queue()
        
        # Fill the token queue
        for token in api_tokens:
            self.available_tokens.put(token)
    
    def llm_hnmr_extraction(self, hnmr_sequence: str) -> str:
        """
        Process a single HNMR sequence using available token
        """
        # Get an available token
        api_token = self.available_tokens.get()
        
        try:
            # Read prompt file
            with open('../prompt/hnmr_process_with_J.txt', 'r') as file:
                prompt = file.read()
            
            # Create OpenAI client with DeepSeek API
            client = Ark(api_key=api_token, timeout=300)
            
            response = client.chat.completions.create(
                model="doubao-seed-1-6-250615",
                messages=[
                    {"role": "system", "content": "You are a chemist specialized in NMR spectrum analysis."},
                    {"role": "user", "content": prompt.format(hnmr_sequence=hnmr_sequence)},
                ],
                thinking={
                    "type": "disabled"
                }
            )
            
            return response.choices[0].message.content
            
        finally:
            # Put the token back for reuse
            self.available_tokens.put(api_token)
    
    def process_multiple_sequences_threaded(self, hnmr_sequences: List[Dict]) -> List[Dict[str, Any]]:
        """
        Process multiple HNMR sequences using threading
        """
        def process_single(entry):
            try:
                result = self.llm_hnmr_extraction(entry['hnmr_sequence'])
                return {
                    'idx': entry['idx'],
                    'sequence': entry['hnmr_sequence'],
                    'result': result,
                    'status': 'success'
                }
            except Exception as e:
                return {
                    'idx': entry['idx'],
                    'sequence': entry['hnmr_sequence'],
                    'result': None,
                    'status': 'error',
                    'error': str(e)
                }
        results = []
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            with tqdm(total=len(hnmr_sequences)) as pbar:
                for result in executor.map(process_single, hnmr_sequences):
                    pbar.update(1)
                    results.append(result)
            # results = list(executor.map(process_single, hnmr_sequences))
        
        return results


api_tokens = [
    "53cf6258-4bee-44ea-95b8-e9f20883d910",
    "9696fedc-e4af-4597-9566-07b20aecb6e7", 
    "662b0ad1-f4c6-4ffd-b3a5-1b4ef3a4fdb6",
    "e9df8562-cbdf-4934-9389-5b58a5bc9422",
    "9d991760-b74e-476f-93b7-42797bd4ea38",
    "a4324b65-79d8-4d2f-9985-950ed2373257",
    "d9f9fa88-b463-4c95-9985-7c39115065a6",
    "b76efa02-4848-4d8a-a5cb-f9f92742a665",
    "67d90425-81e9-4a9a-b0f1-794246419166",
    "9b03d668-5593-4442-b723-44acd5fac92b",
    "3abfce01-7274-4851-ae66-8bbe9046fc59",
    "9249302b-e213-4253-8404-49b2f031eef6",
    "0551e3a2-f71c-4f0a-813a-6395b04f1bb7",
    "fae4fb60-fa53-4002-b568-39824063fbfd",
    "6c30c5bf-82d4-43de-9100-1bbc3b09efbd",
    "12fa352d-00ee-4393-9be6-ff21795171c0",
    "306de6a6-09bc-416f-8914-26e0b705ae83",
    "ac016b87-3cd8-42bc-ab20-4ae1b17763f0",
    "a8133c14-46d4-4c8b-9caf-79488dc234dc",
    "f28baaef-ad2e-4e03-bcf9-d6ac12f2df2c",
    "dbeb1c64-a066-4852-bc23-90802d85cbdd",
    "2050f831-b480-4d73-bceb-4db7a41d8d98",
    "86b79b89-eb1b-467c-872c-4535a5f0a73c",
    "08cfb1f0-fb0c-4864-9b1c-e0eb2cea1ed9",
    "60755744-e205-46e3-bfee-ce183c42eefa",
    "32292cc7-b4c5-46d5-b2ff-a58414e30ef0",
    "59e839a3-dc32-4056-927f-9ddd66f20e1b",
    "930479f3-dabf-473d-9743-4908bcb62c8c",
    "59ae9e59-9699-4615-a91a-0de17e6dfd9f",
    "23a4df1c-ac9e-4664-8330-14e069b4a1aa",
    "36adb71b-d95a-4309-8060-779359bd872a",
    "aa3985ae-488a-41f4-a487-77d2be3da9c4",
    "ce8d4aab-c89a-499c-b524-22febf3a2b86",
    "ff2e1b86-06a0-4d9b-ae4c-bd627bd63aff",
    "9a157d9a-848c-4aae-a8f2-73b18411f3ee",
    "fc1e06bd-fe3f-4a9f-819c-6dd01cc93a04",
    "98096a71-060f-4f6e-ad6d-74f3da73c620",
    "164ca276-0d20-4255-9305-f4ea38683b96",
    "11df3abc-0cef-408e-8ede-e88710251090",
    "2bcf17cd-179e-4e6f-9a86-65a54365734e"
]

hnmr_sequences = []
for i in range(len(NMRBank)):
    hnmr_sequences.append({
        'idx': i,
        'hnmr_sequence': NMRBank['1H NMR chemical shifts'].iloc[i],
    })
# Method 1: Using threading with queue-based token management (Recommended)
print("Using threading with queue-based token management...")
thread_processor = DeepSeekThreadProcessor(api_tokens, max_workers=len(api_tokens))
thread_results = thread_processor.process_multiple_sequences_threaded(hnmr_sequences)


In [ ]:
import json
import numpy as np

def convert(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

with open('../../raw_data/NMRBank/NMRBank_hnmr.json', 'w', encoding='utf-8') as f:
    json.dump(thread_results, f, indent=4, default=convert)

# Dataset Curation
## Format Conversion

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
import random
from tqdm import tqdm
import ast
import numpy as np
data_list = []
error_list = []
error_idx_list = []
filter_list = []
with open('../../raw_data/NMRBank/NMRBank_hnmr.json', 'r') as f:
    thread_results = json.load(f)
for i in tqdm(range(len(thread_results))):
    try:
        smiles = NMRBank.iloc[i]['Standardized SMILES']
        mol = Chem.MolFromSmiles(smiles)
        molecular_formula = rdMolDescriptors.CalcMolFormula(mol)
        h_nmr_peaks = []
        c_nmr_peaks = []
        # Process CNMR data
        cnmr = NMRBank.iloc[i]['13C NMR chemical shifts']
        cnmr = [s.strip().replace(' ', '') for s in cnmr.split(',') if s.strip()]
        for peak in cnmr:
            if '−' in peak:
                if peak.split('−')[0]:
                    peak_value = (float(peak.split('−')[0]) + float(peak.split('−')[1])) / 2
                else:
                    peak_value = float(peak.replace('−', '-'))
            elif '–' in peak:
                if peak.split('–')[0]:
                    peak_value = (float(peak.split('–')[0]) + float(peak.split('–')[1])) / 2
                else:
                    peak_value = float(peak.replace('–', '-'))
            else:
                peak_value = float(peak)
            c_nmr_peaks.append({'delta (ppm)': peak_value, 'integral': None, 'intensity': None, 'width (ppm)': None})
        
        # Process HNMR data
        hnmr = thread_results[i]['result_converted']
        if hnmr is None:
            raise ValueError("HNMR extraction failed")
        for h in hnmr:
            if h['value'] == None:
                if h['rangeMin'] != None and h['rangeMax'] != None:
                    h['value'] = (h['rangeMin'] + h['rangeMax']) / 2
                else:
                    continue
            if h['rangeMin'] == None or h['rangeMax'] == None:
                if h['value'] != None:
                    h['rangeMin'] = h['value']
                    h['rangeMax'] = h['value']
                else:
                    continue
            h_nmr_peaks.append({'category': h['type'], 'centroid': h['value'], 'delta': h['value'], 'j_values': None, 'nH': int(h['intensity']) if h['intensity'] is not None else 1, 'rangeMax': h['rangeMax'], 'rangeMin': h['rangeMin']})
        # Compute num_C and num_H
        num_C = sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == 'C')
        num_H = sum(atom.GetTotalNumHs() for atom in mol.GetAtoms())
        data_list.append({
            'idx': i,
            'molecular_formula': molecular_formula,
            'num_C': num_C,
            'num_H': num_H,
            'c_nmr_peaks': c_nmr_peaks,
            'h_nmr_peaks': h_nmr_peaks,
            'num_h_peaks': sum(h['nH'] for h in h_nmr_peaks),
            'num_c_peaks': len(c_nmr_peaks),
            'smiles': smiles
        })
    except Exception as e:
        error_list.append(e)
        error_idx_list.append(i)



## NMR2Mol Dataset Generation

In [ ]:
from nmr_rise.utils.data import mol2nmr_target_generation, calculate_rmse, conformation_construction, filter_invalid_smiles
from datasets import DatasetDict
import os
from datasets import load_from_disk, Dataset
dataset = Dataset.from_list(data_list)
print("Dataset size before filtering:", len(dataset))
dataset = dataset.filter(filter_invalid_smiles, num_proc=16)
print("Dataset size after filtering:", len(dataset))
# split dataset into train, val, test with 80% train, 10% val, 10% test
def cal_max_shift(entry):
    cnmr_values = [peak['delta (ppm)'] for peak in entry['c_nmr_peaks']]
    hnmr_values = [peak['delta'] for peak in entry['h_nmr_peaks']]
    max_c_shift = max(cnmr_values) if cnmr_values else 0
    max_h_shift = max(hnmr_values) if hnmr_values else 0
    min_c_shift = min(cnmr_values) if cnmr_values else 0
    min_h_shift = min(hnmr_values) if hnmr_values else 0
    entry['max_c_shift'] = max_c_shift
    entry['max_h_shift'] = max_h_shift
    entry['min_c_shift'] = min_c_shift
    entry['min_h_shift'] = min_h_shift
    return entry
dataset = dataset.map(cal_max_shift, num_proc=16)
# print("Dataset size before filtering:", len(dataset['train']))
dataset = dataset.filter(lambda x: x['max_c_shift'] <= 250 and x['max_h_shift'] <= 15, num_proc=16)
dataset = dataset.filter(lambda x: x['min_c_shift'] >= -20 and x['min_h_shift'] >= -2, num_proc=16)
print("Dataset size after filtering:", len(dataset))
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split['train']
test_valid_dataset = train_test_split['test']
val_test_split = test_valid_dataset.train_test_split(test_size=0.5, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']
train_dataset.remove_columns(['atoms', 'coordinates', 'atoms_target', 'atoms_target_mask', 'smiles_with_hydrogens', 'inchikey', 'is_converted'])
val_dataset.remove_columns(['atoms', 'coordinates', 'atoms_target', 'atoms_target_mask', 'smiles_with_hydrogens', 'inchikey', 'is_converted'])
test_dataset.remove_columns(['atoms', 'coordinates', 'atoms_target', 'atoms_target_mask', 'smiles_with_hydrogens', 'inchikey', 'is_converted'])
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})
dataset.save_to_disk('../../data/NMRBank/full')

## Mol2NMR Dataset Generation

In [ ]:
from datasets import load_from_disk
from nmr_rise.utils.data import conformation_construction, filter_invalid_entry
dataset = load_from_disk('../../data/NMRBank/full')
dataset.cleanup_cache_files()
for split in ['train', 'validation', 'test']:
    dataset[split] = dataset[split].map(conformation_construction, num_proc=16)
    dataset[split] = dataset[split].filter(filter_invalid_entry, num_proc=16)
dataset.save_to_disk('../../data/NMRBank/mol2nmr/full_cc')

## MolRef Dataset Generation

In [ ]:
import os
from datasets import load_from_disk
import json
import ast
import copy
def load_nmr2mol_pred_results(dir_path, filename):
    with open(os.path.join(dir_path, filename), 'r') as f:
        return {item['true']: str(item['pred']) for item in json.load(f)}

def add_candidates(entry, lookup, validity_check=True):
    candidates = ast.literal_eval(lookup.get(entry['smiles'], ""))

    if validity_check:
        valid_candidates = []
        for cand in candidates:
            confs = conformation_construction({'smiles': copy.deepcopy(cand)})
            if confs['is_converted']:
                valid_candidates.append(cand)
        entry['candidates'] = valid_candidates
    else:
        entry['candidates'] = candidates
    return entry

def molref_data_gen(data_path, dataset_name, nmr2mol_pred_dir):
    dataset = load_from_disk(os.path.join(data_path, dataset_name))
    nmr2mol_pred_path = os.path.join(data_path, nmr2mol_pred_dir)
    lookups = {
        'train': load_nmr2mol_pred_results(nmr2mol_pred_path, 'train_split_result.json'),
        'validation': load_nmr2mol_pred_results(nmr2mol_pred_path, 'valid_split_result.json'),
        'test': load_nmr2mol_pred_results(nmr2mol_pred_path, 'test_split_result.json')
    }
    for split in ['train', 'validation', 'test']:
        dataset[split] = dataset[split].map(lambda ex: add_candidates(ex, lookups[split], validity_check=False), num_proc=16)
    return dataset

In [ ]:
dataset = molref_data_gen('../../data/NMRBank', 'full', 'nmr2mol_pred')

In [ ]:
import random
from ast import literal_eval
def candidate_aug(batch, aug_num=5):
    """
    Batch version of candidate augmentation function.
    For each example in the batch, samples aug_num candidates from the 'candidates' list,
    then expands each example into multiple rows - one per sampled candidate.
    
    Args:
        batch (dict): A batch of examples as lists for each column.
        aug_num (int): Number of candidates to sample and expand per example.
    
    Returns:
        dict: Augmented batch with each example expanded into multiple rows.
    """
    aug_rows = {
        k: [] for k in batch if k != 'candidates'
    }
    aug_rows['candidate'] = []
    aug_rows['aug_id'] = []
    
    for i, candidates in enumerate(batch['candidates']):
        # Sample candidates with or without replacement
        sampled_candidates = random.sample(candidates, aug_num)
        
        for j, candidate in enumerate(sampled_candidates):
            # Copy all columns except 'candidates'
            for key in batch:
                if key != 'candidates':
                    aug_rows[key].append(batch[key][i])
            aug_rows['candidate'].append(candidate)
            aug_rows['aug_id'].append(f"aug_{batch.get('idx', [''])[i]}_{j}")

    return aug_rows

def dataset_aug_molref(dataset, aug_num=5, aug_max=False):
    """
    Augments the dataset for training the MolRef model by expanding each example into multiple rows based on candidates.
    
    Args:
        dataset (datasets.Dataset): HuggingFace dataset to augment.
        aug_num (int): Number of augmented samples per example.
    
    Returns:
        datasets.Dataset: Augmented dataset with expanded rows and extra columns.
    """
    return dataset.map(
        lambda batch: candidate_aug(batch, aug_num=len(batch['candidates'][0]) if aug_max else min(aug_num, len(batch['candidates'][0]))),
        batched=True,
        batch_size=1,  # Process one example at a time for expansion
        remove_columns=['candidates'],
        num_proc=os.cpu_count()
    )


In [ ]:
from datasets import DatasetDict
for aug_num in [1, 3, 5, 10]:
    shuffle_seed = 42
    train_aug = dataset_aug_molref(dataset['train'], aug_num=aug_num, aug_max=False)
    val_aug = dataset_aug_molref(dataset['validation'], aug_num=aug_num, aug_max=False)
    test_aug = dataset_aug_molref(dataset['test'], aug_num=aug_num, aug_max=False)
    train_aug = train_aug.shuffle(seed=shuffle_seed)
    val_aug = val_aug.shuffle(seed=shuffle_seed)
    test_aug = test_aug.shuffle(seed=shuffle_seed)
    aug_dataset = DatasetDict({
        'train': train_aug,
        'validation': val_aug,
        'test': test_aug
    })
    aug_dataset.save_to_disk(os.path.join('../../data/NMRBank', f'revision_{aug_num}'))

# Evaluation Dataset Generation

In [ ]:
from datasets import load_from_disk
import random

test_dataset = load_from_disk('../../data/NMRBank/full')['test']

# randomly select 1000 samples from test_dataset
random.seed(42)
selected_indices = random.sample(range(len(test_dataset)), 10000)
selected_samples = test_dataset.select(selected_indices)
selected_samples.save_to_disk('../../data/NMRBank/NMRBank-10000')

selected_indices = random.sample(range(len(selected_samples)), 1000)
selected_samples = selected_samples.select(selected_indices)
selected_samples.save_to_disk('../../data/NMRBank/NMRBank-1000')